# 03 — Seattle GCCP: Causal-Weighted Feature Similarity

This notebook develops the first GCCP implementation for the Seattle housing experiment.

The goal is to modify only the feature-similarity component of GeoSIMCP while preserving the same:

- data splits;
- XGBoost predictor;
- geographic distance;
- Gaussian kernel;
- weighted conformal quantile;
- bandwidth and lambda search grids;
- Inner Validation parameter-selection protocol.

For the Inner experiment, causal feature weights are estimated from **Inner Fit only** using GCCM.

In [1]:
# Chapter 1 — GCCP preflight

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

GCCP_ROOT = PROJECT_ROOT / "GCCP_Seattle"
OUTPUT_DIR = GCCP_ROOT / "outputs"
GCCM_DIR = OUTPUT_DIR / "gccm"

SEATTLE_CSV = (
    PROJECT_ROOT
    / "example"
    / "seattle_sample_3k.csv"
)

OUTER_SPLIT_PATH = (
    OUTPUT_DIR
    / "seattle_split_seed42.npz"
)

INNER_SPLIT_PATH = (
    OUTPUT_DIR
    / "seattle_inner_split_seed42.npz"
)

GCCM_WEIGHT_PATH = (
    GCCM_DIR
    / "seattle_gccm_causal_weights_inner_fit_v1.csv"
)

paths = {
    "Seattle CSV": SEATTLE_CSV,
    "Outer split": OUTER_SPLIT_PATH,
    "Inner split": INNER_SPLIT_PATH,
    "Inner GCCM weights": GCCM_WEIGHT_PATH,
}

print("===== GCCP PREFLIGHT =====")

for name, path in paths.items():
    print(
        f"{name:22s}:",
        "FOUND" if path.exists() else "MISSING"
    )
    print("   ", path)

outer_npz = np.load(OUTER_SPLIT_PATH)
inner_npz = np.load(INNER_SPLIT_PATH)

print("\nOuter split keys:")
print(outer_npz.files)

print("\nInner split keys:")
print(inner_npz.files)

gccm_weights_inner = pd.read_csv(GCCM_WEIGHT_PATH)

print("\nInner GCCM weights:")
display(
    gccm_weights_inner[
        ["feature", "causal_strength", "causal_weight"]
    ].round(6)
)

print(
    "\nWeight sum:",
    gccm_weights_inner["causal_weight"].sum()
)

===== GCCP PREFLIGHT =====
Seattle CSV           : FOUND
    E:\GeoSIMCP_Reproduce\geoconformal-main\example\seattle_sample_3k.csv
Outer split           : FOUND
    E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_split_seed42.npz
Inner split           : FOUND
    E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_inner_split_seed42.npz
Inner GCCM weights    : FOUND
    E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_causal_weights_inner_fit_v1.csv

Outer split keys:
['train_indices', 'calibration_indices', 'test_indices', 'random_state']

Inner split keys:
['inner_fit_indices', 'inner_calibration_indices', 'inner_validation_indices', 'random_state']

Inner GCCM weights:


,feature,causal_strength,causal_weight
0,bathrooms,0.193219,0.158578
1,sqft_living,0.267773,0.219766
2,sqft_lot,0.201288,0.165200
3,grade,0.315271,0.258749
4,condition,0.089394,0.073367
5,waterfront,0.000000,0.000000
6,view,0.151501,0.124340
7,age,0.000000,0.000000



Weight sum: 0.9999999999999999


## Chapter 2 — Restore the frozen Seattle experiment

The GCCP experiment reuses exactly the same Seattle data preprocessing,
Outer split, Inner split, predictor variables, and XGBoost configuration
as the frozen baseline experiment.

No new random data split is generated in this notebook.

The eight non-spatial attributes are used for similarity calculation,
while the XGBoost predictor uses the eight attributes together with
UTM coordinates.

In [2]:
# Chapter 2.1 — Restore Seattle data and frozen splits

import numpy as np
import pandas as pd

from pyproj import Transformer

# ---------------------------------------------------------
# 1. Load original Seattle data
# ---------------------------------------------------------

df = pd.read_csv(SEATTLE_CSV)

ASpatial_FEATURES = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]

SPATIAL_FEATURES = [
    "UTM_X",
    "UTM_Y",
]

MODEL_FEATURES = (
    ASpatial_FEATURES
    + SPATIAL_FEATURES
)

# Target used throughout the baseline experiment
y_all = (
    10 ** df["log_price"]
    / 10000.0
).astype(float)

X_aspatial_all = (
    df[ASpatial_FEATURES]
    .astype(float)
    .copy()
)

X_model_all = (
    df[MODEL_FEATURES]
    .astype(float)
    .copy()
)

# ---------------------------------------------------------
# 2. Restore geographic coordinates exactly once
#    UTM Zone 10N -> WGS84
# ---------------------------------------------------------

transformer = Transformer.from_crs(
    "EPSG:32610",
    "EPSG:4326",
    always_xy=True
)

lon_all, lat_all = transformer.transform(
    df["UTM_X"].to_numpy(dtype=float),
    df["UTM_Y"].to_numpy(dtype=float)
)

loc_all = np.column_stack([
    lon_all,
    lat_all
])

# ---------------------------------------------------------
# 3. Restore Outer split
# ---------------------------------------------------------

outer_train_indices = np.asarray(
    outer_npz["train_indices"],
    dtype=int
)

outer_calibration_indices = np.asarray(
    outer_npz["calibration_indices"],
    dtype=int
)

outer_test_indices = np.asarray(
    outer_npz["test_indices"],
    dtype=int
)

# ---------------------------------------------------------
# 4. Restore Inner split
#
# Detect whether saved Inner indices are global indices
# or positions inside Outer Train.
# ---------------------------------------------------------

inner_fit_raw = np.asarray(
    inner_npz["inner_fit_indices"],
    dtype=int
)

inner_cal_raw = np.asarray(
    inner_npz["inner_calibration_indices"],
    dtype=int
)

inner_val_raw = np.asarray(
    inner_npz["inner_validation_indices"],
    dtype=int
)

outer_train_set = set(
    outer_train_indices.tolist()
)

all_inner_raw = np.concatenate([
    inner_fit_raw,
    inner_cal_raw,
    inner_val_raw
])

if set(all_inner_raw.tolist()).issubset(
    outer_train_set
):
    print("Inner indices detected as GLOBAL row indices.")

    inner_fit_indices = inner_fit_raw
    inner_calibration_indices = inner_cal_raw
    inner_validation_indices = inner_val_raw

elif (
    all_inner_raw.min() >= 0
    and all_inner_raw.max() < len(outer_train_indices)
):
    print("Inner indices detected as positions within Outer Train.")

    inner_fit_indices = outer_train_indices[
        inner_fit_raw
    ]

    inner_calibration_indices = outer_train_indices[
        inner_cal_raw
    ]

    inner_validation_indices = outer_train_indices[
        inner_val_raw
    ]

else:
    raise ValueError(
        "Could not determine the meaning of saved Inner indices."
    )

# ---------------------------------------------------------
# 5. Integrity checks
# ---------------------------------------------------------

assert len(outer_train_indices) == 2100
assert len(outer_calibration_indices) == 450
assert len(outer_test_indices) == 450

assert len(inner_fit_indices) == 1470
assert len(inner_calibration_indices) == 315
assert len(inner_validation_indices) == 315

assert set(inner_fit_indices).isdisjoint(
    inner_calibration_indices
)

assert set(inner_fit_indices).isdisjoint(
    inner_validation_indices
)

assert set(inner_calibration_indices).isdisjoint(
    inner_validation_indices
)

inner_union = set(
    np.concatenate([
        inner_fit_indices,
        inner_calibration_indices,
        inner_validation_indices
    ]).tolist()
)

assert inner_union == outer_train_set

print("\n===== RESTORED SPLITS =====")

print(
    "Outer:",
    len(outer_train_indices),
    len(outer_calibration_indices),
    len(outer_test_indices)
)

print(
    "Inner:",
    len(inner_fit_indices),
    len(inner_calibration_indices),
    len(inner_validation_indices)
)

print("\nDataset shape:", df.shape)
print("X model shape:", X_model_all.shape)
print("X similarity shape:", X_aspatial_all.shape)
print("Coordinates shape:", loc_all.shape)

print(
    "\nTarget range:",
    y_all.min(),
    "to",
    y_all.max()
)

print("\n✓ Frozen Seattle data and splits restored.")

Inner indices detected as GLOBAL row indices.

===== RESTORED SPLITS =====
Outer: 2100 450 450
Inner: 1470 315 315

Dataset shape: (3000, 12)
X model shape: (3000, 10)
X similarity shape: (3000, 8)
Coordinates shape: (3000, 2)

Target range: 8.2 to 363.5000000000002

✓ Frozen Seattle data and splits restored.


In [3]:
# Chapter 2.2 — Restore the frozen Inner XGBoost predictor

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

X_inner_fit = X_model_all.iloc[
    inner_fit_indices
].copy()

X_inner_cal = X_model_all.iloc[
    inner_calibration_indices
].copy()

X_inner_val = X_model_all.iloc[
    inner_validation_indices
].copy()

y_inner_fit = y_all.iloc[
    inner_fit_indices
].copy()

y_inner_cal = y_all.iloc[
    inner_calibration_indices
].copy()

y_inner_val = y_all.iloc[
    inner_validation_indices
].copy()

X_aspatial_inner_cal = X_aspatial_all.iloc[
    inner_calibration_indices
].copy()

X_aspatial_inner_val = X_aspatial_all.iloc[
    inner_validation_indices
].copy()

loc_inner_cal = loc_all[
    inner_calibration_indices
]

loc_inner_val = loc_all[
    inner_validation_indices
]

XGBOOST_PARAMS = {
    "n_estimators": 500,
    "max_depth": 3,
    "min_child_weight": 1.0,
    "colsample_bytree": 1.0,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}

model_inner = XGBRegressor(
    **XGBOOST_PARAMS
)

model_inner.fit(
    X_inner_fit,
    y_inner_fit
)

pred_inner_val = model_inner.predict(
    X_inner_val
)

mae_inner = mean_absolute_error(
    y_inner_val,
    pred_inner_val
)

rmse_inner = root_mean_squared_error(
    y_inner_val,
    pred_inner_val
)

r2_inner = r2_score(
    y_inner_val,
    pred_inner_val
)

print("===== INNER PREDICTOR REPRODUCTION =====")

print(f"MAE  : {mae_inner:.4f}")
print(f"RMSE : {rmse_inner:.4f}")
print(f"R²   : {r2_inner:.4f}")

print("\nExpected frozen baseline:")
print("MAE  ≈ 9.7474")
print("RMSE ≈ 17.5917")
print("R²   ≈ 0.7941")

===== INNER PREDICTOR REPRODUCTION =====
MAE  : 9.7474
RMSE : 17.5917
R²   : 0.7941

Expected frozen baseline:
MAE  ≈ 9.7474
RMSE ≈ 17.5917
R²   ≈ 0.7941


## Chapter 3 — GCCP causal feature similarity

GeoSIMCP-EUC measures feature similarity using the Euclidean distance between standardized non-spatial attributes.

GCCP retains the same standardized feature space but introduces GCCM-derived feature importance.

For calibration sample \(i\) and target sample \(t\),

$$
d_{\mathrm{causal}}(i,t)
=
\sqrt{
\sum_{j=1}^{p}
p w_j
(z_{ij}-z_{tj})^2
},
$$

where:

- \(p=8\) is the number of non-spatial attributes;
- \(w_j\) is the normalized GCCM causal weight;
- \(z_j\) is standardized using statistics fitted on the calibration set only.

The factor \(p\) makes the distance scale comparable with ordinary Euclidean feature distance.

If all features have equal causal weight,

$$
w_j=\frac{1}{p},
$$

then

$$
p w_j=1,
$$

and therefore

$$
d_{\mathrm{causal}}
=
d_{\mathrm{EUC}}.
$$

This property allows GCCP to differ from GeoSIMCP-EUC only through the GCCM-derived feature weighting.

In [4]:
# Chapter 3.1 — Align and inspect GCCM causal weights

feature_order = ASpatial_FEATURES.copy()

weights_aligned = (
    gccm_weights_inner
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

causal_weight_vector = (
    weights_aligned["causal_weight"]
    .to_numpy(dtype=float)
)

p_features = len(feature_order)

# Scale-preserving distance weights
causal_distance_weights = (
    p_features * causal_weight_vector
)

weight_check = pd.DataFrame({
    "feature": feature_order,
    "causal_weight": causal_weight_vector,
    "distance_weight_pwj": causal_distance_weights
})

display(weight_check.round(6))

print(
    "Normalized causal weight sum:",
    causal_weight_vector.sum()
)

print(
    "Distance-weight sum:",
    causal_distance_weights.sum()
)

assert np.isclose(
    causal_weight_vector.sum(),
    1.0
)

assert np.isclose(
    causal_distance_weights.sum(),
    p_features
)

print("\n✓ GCCM weights aligned with the 8 GeoSIMCP features.")

,feature,causal_weight,distance_weight_pwj
0,bathrooms,0.158578,1.268626
1,sqft_living,0.219766,1.758130
2,sqft_lot,0.165200,1.321602
3,grade,0.258749,2.069990
4,condition,0.073367,0.586936
5,waterfront,0.000000,0.000000
6,view,0.124340,0.994716
7,age,0.000000,0.000000


Normalized causal weight sum: 0.9999999999999999
Distance-weight sum: 7.999999999999999

✓ GCCM weights aligned with the 8 GeoSIMCP features.


In [5]:
# Chapter 3.2 — Standardize similarity features using Calibration only

from sklearn.preprocessing import StandardScaler

similarity_scaler_inner = StandardScaler()

Z_inner_cal = similarity_scaler_inner.fit_transform(
    X_aspatial_inner_cal
)

Z_inner_val = similarity_scaler_inner.transform(
    X_aspatial_inner_val
)

print("Standardized shapes:")
print("Calibration:", Z_inner_cal.shape)
print("Validation :", Z_inner_val.shape)

print("\nCalibration standardized means:")
print(
    np.round(
        Z_inner_cal.mean(axis=0),
        10
    )
)

print("\nCalibration standardized std:")
print(
    np.round(
        Z_inner_cal.std(axis=0),
        10
    )
)

assert Z_inner_cal.shape == (315, 8)
assert Z_inner_val.shape == (315, 8)

assert np.isfinite(Z_inner_cal).all()
assert np.isfinite(Z_inner_val).all()

print("\n✓ Similarity features standardized from Inner Calibration only.")

Standardized shapes:
Calibration: (315, 8)
Validation : (315, 8)

Calibration standardized means:
[-0. -0. -0. -0. -0.  0.  0. -0.]

Calibration standardized std:
[1. 1. 1. 1. 1. 1. 1. 1.]

✓ Similarity features standardized from Inner Calibration only.


In [6]:
# Chapter 3.3 — Feature-distance functions

from scipy.spatial.distance import cdist


def euclidean_feature_distance(
    Z_cal,
    Z_test
):
    """
    Ordinary standardized Euclidean feature distance.

    Returns:
        shape = (n_test, n_cal)
    """
    return cdist(
        Z_test,
        Z_cal,
        metric="euclidean"
    )


def causal_feature_distance(
    Z_cal,
    Z_test,
    normalized_causal_weights
):
    """
    GCCP causal-weighted Euclidean feature distance.

    d^2 = sum_j p * w_j * (z_cal_j - z_test_j)^2
    """

    weights = np.asarray(
        normalized_causal_weights,
        dtype=float
    )

    p = Z_cal.shape[1]

    assert len(weights) == p
    assert np.all(weights >= 0)
    assert np.isclose(weights.sum(), 1.0)

    scale = np.sqrt(
        p * weights
    )

    Z_cal_weighted = (
        Z_cal * scale
    )

    Z_test_weighted = (
        Z_test * scale
    )

    return cdist(
        Z_test_weighted,
        Z_cal_weighted,
        metric="euclidean"
    )

In [7]:
# Chapter 3.4 — Degeneracy test:
# equal causal weights must reproduce GeoSIMCP-EUC

d_euc = euclidean_feature_distance(
    Z_inner_cal,
    Z_inner_val
)

equal_weights = np.ones(
    p_features
) / p_features

d_causal_equal = causal_feature_distance(
    Z_inner_cal,
    Z_inner_val,
    equal_weights
)

max_abs_difference = np.max(
    np.abs(
        d_euc
        - d_causal_equal
    )
)

print(
    "Euclidean distance matrix shape:",
    d_euc.shape
)

print(
    "Equal-weight causal matrix shape:",
    d_causal_equal.shape
)

print(
    "\nMaximum absolute difference:",
    f"{max_abs_difference:.3e}"
)

assert np.allclose(
    d_euc,
    d_causal_equal,
    rtol=1e-12,
    atol=1e-12
)

print(
    "\n✓ Equal-weight GCCP exactly reproduces GeoSIMCP-EUC."
)

Euclidean distance matrix shape: (315, 315)
Equal-weight causal matrix shape: (315, 315)

Maximum absolute difference: 0.000e+00

✓ Equal-weight GCCP exactly reproduces GeoSIMCP-EUC.


In [8]:
# Chapter 3.5 — Inner GCCP causal feature distance

d_causal_inner = causal_feature_distance(
    Z_inner_cal,
    Z_inner_val,
    causal_weight_vector
)

print(
    "Causal distance matrix shape:",
    d_causal_inner.shape
)

print(
    "\nEuclidean feature distance:"
)

print(
    "  mean   =",
    d_euc.mean()
)

print(
    "  median =",
    np.median(d_euc)
)

print(
    "  min    =",
    d_euc.min()
)

print(
    "  max    =",
    d_euc.max()
)

print(
    "\nGCCM-weighted causal feature distance:"
)

print(
    "  mean   =",
    d_causal_inner.mean()
)

print(
    "  median =",
    np.median(d_causal_inner)
)

print(
    "  min    =",
    d_causal_inner.min()
)

print(
    "  max    =",
    d_causal_inner.max()
)

print(
    "\nCorrelation between EUC and causal distance:",
    np.corrcoef(
        d_euc.ravel(),
        d_causal_inner.ravel()
    )[0, 1]
)

assert d_causal_inner.shape == (
    len(inner_validation_indices),
    len(inner_calibration_indices)
)

assert np.isfinite(
    d_causal_inner
).all()

print(
    "\n✓ Inner causal feature-distance matrix ready."
)

Causal distance matrix shape: (315, 315)

Euclidean feature distance:
  mean   = 3.623143318241314
  median = 3.3602787930688502
  min    = 0.04969003616681274
  max    = 18.162357876090276

GCCM-weighted causal feature distance:
  mean   = 3.728294782420086
  median = 3.3654183260107016
  min    = 0.019912666432338955
  max    = 17.403218056712678

Correlation between EUC and causal distance: 0.879226334382304

✓ Inner causal feature-distance matrix ready.


## Chapter 4 — GCCP conformal calibration

GCCP retains the same conformal calibration mechanism used by the
GeoCP and GeoSIMCP baselines.

For each validation point, calibration residuals are weighted according
to the joint geographic-causal distance.

The Gaussian weight is

$$
K(d)
=
\exp\left[
-\frac{1}{2}
\left(
\frac{d}{h}
\right)^2
\right],
$$

where \(h\) is the bandwidth.

The joint distance is

$$
d_{\mathrm{joint}}
=
\sqrt{
\lambda d_{\mathrm{geo}}^2
+
(1-\lambda)d_{\mathrm{causal}}^2
}.
$$

The weighted empirical quantile follows the same implementation as the
frozen GeoCP / GeoSIMCP baselines.

When \(\lambda=1\), GCCP must reduce exactly to GeoCP.

In [9]:
# Chapter 4.1 — Inner calibration nonconformity scores

ALPHA = 0.10

pred_inner_cal = model_inner.predict(
    X_inner_cal
)

calibration_scores_inner = np.abs(
    y_inner_cal.to_numpy(dtype=float)
    - pred_inner_cal
)

print("Inner Calibration residuals:")
print("  n      =", len(calibration_scores_inner))
print("  mean   =", calibration_scores_inner.mean())
print("  median =", np.median(calibration_scores_inner))
print("  max    =", calibration_scores_inner.max())

assert len(calibration_scores_inner) == 315
assert np.isfinite(calibration_scores_inner).all()

print("\n✓ Inner calibration scores ready.")

Inner Calibration residuals:
  n      = 315
  mean   = 8.020875772259425
  median = 4.642739105224571
  max    = 100.3107910156248

✓ Inner calibration scores ready.


In [10]:
# Chapter 4.2 — Geographic distance
# Preserve the frozen GeoCP / GeoSIMCP implementation

from scipy.spatial.distance import cdist

d_geo_inner = cdist(
    loc_inner_val,
    loc_inner_cal,
    metric="euclidean"
)

print("Geographic distance matrix:")
print("  shape  =", d_geo_inner.shape)
print("  mean   =", d_geo_inner.mean())
print("  median =", np.median(d_geo_inner))
print("  min    =", d_geo_inner.min())
print("  max    =", d_geo_inner.max())

assert d_geo_inner.shape == (315, 315)
assert np.isfinite(d_geo_inner).all()

print("\n✓ Inner geographic distances ready.")

Geographic distance matrix:
  shape  = (315, 315)
  mean   = 0.19526686134993934
  median = 0.19022463037156517
  min    = 0.0002000000000066393
  max    = 0.5476851376475352

✓ Inner geographic distances ready.


In [11]:
# Chapter 4.3 — Author-style weighted quantile

def weighted_quantile_author(
    scores,
    weights,
    alpha=0.10
):
    scores = np.asarray(
        scores,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    sorted_indices = np.argsort(scores)

    scores_sorted = scores[
        sorted_indices
    ]

    weights_sorted = weights[
        sorted_indices
    ]

    total_weight = weights_sorted.sum()

    if (
        not np.isfinite(total_weight)
        or total_weight <= 0
    ):
        return np.nan

    cumsum_weights = np.cumsum(
        weights_sorted
    )

    normalized_cumsum = (
        cumsum_weights
        / total_weight
    )

    idx = np.searchsorted(
        normalized_cumsum,
        1 - alpha
    )

    idx = min(
        idx,
        len(scores_sorted) - 1
    )

    return scores_sorted[idx]

In [12]:
# Chapter 4.4 — Evaluate one GCCP parameter combination

def interval_score(
    y_true,
    lower,
    upper,
    alpha=0.10
):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower

    below = (
        (lower - y_true)
        * (y_true < lower)
    )

    above = (
        (y_true - upper)
        * (y_true > upper)
    )

    return (
        width
        + (2 / alpha) * below
        + (2 / alpha) * above
    )


def evaluate_gccp_inner(
    bandwidth,
    lambda_weight,
    d_geo,
    d_causal,
    calibration_scores,
    validation_predictions,
    y_validation,
    alpha=0.10
):
    # Joint geographic-causal distance
    d_joint = np.sqrt(
        lambda_weight * d_geo**2
        +
        (1 - lambda_weight)
        * d_causal**2
    )

    # Gaussian kernel
    weights = np.exp(
        -0.5
        * (d_joint / bandwidth)**2
    )

    n_val = d_joint.shape[0]

    q = np.empty(
        n_val,
        dtype=float
    )

    for i in range(n_val):
        q[i] = weighted_quantile_author(
            calibration_scores,
            weights[i],
            alpha=alpha
        )

    lower = (
        validation_predictions - q
    )

    upper = (
        validation_predictions + q
    )

    y_true = np.asarray(
        y_validation,
        dtype=float
    )

    coverage = np.mean(
        (y_true >= lower)
        &
        (y_true <= upper)
    )

    mean_width = np.mean(
        upper - lower
    )

    mean_score = np.mean(
        interval_score(
            y_true,
            lower,
            upper,
            alpha=alpha
        )
    )

    return {
        "coverage": coverage,
        "mean_width": mean_width,
        "interval_score": mean_score,
        "lower": lower,
        "upper": upper,
        "q": q
    }

In [13]:
# Chapter 4.5 — GCCP -> GeoCP degeneracy test

geocp_reproduction = evaluate_gccp_inner(
    bandwidth=0.1,
    lambda_weight=1.0,
    d_geo=d_geo_inner,
    d_causal=d_causal_inner,
    calibration_scores=calibration_scores_inner,
    validation_predictions=pred_inner_val,
    y_validation=y_inner_val,
    alpha=ALPHA
)

print("===== GCCP λ=1 → GeoCP CHECK =====")

print(
    f"Coverage       : "
    f"{geocp_reproduction['coverage']:.4f}"
)

print(
    f"Mean width     : "
    f"{geocp_reproduction['mean_width']:.4f}"
)

print(
    f"Interval Score : "
    f"{geocp_reproduction['interval_score']:.4f}"
)

print("\nFrozen Inner GeoCP expected:")
print("Coverage       ≈ 0.8571")
print("Mean width     ≈ 33.6271")
print("Interval Score ≈ 80.6478")

===== GCCP λ=1 → GeoCP CHECK =====
Coverage       : 0.8571
Mean width     : 33.6271
Interval Score : 80.6478

Frozen Inner GeoCP expected:
Coverage       ≈ 0.8571
Mean width     ≈ 33.6271
Interval Score ≈ 80.6478


## Chapter 5 — Inner GCCP parameter tuning

GCCP uses the same hyperparameter grids as the frozen GeoSIMCP baseline:

$$
h \in \operatorname{linspace}(0.1,5.0,20)
$$

and

$$
\lambda \in \operatorname{linspace}(0,1,20).
$$

All 400 parameter combinations are evaluated using Inner Calibration
and Inner Validation only.

The selection rule is frozen before inspecting the Outer Test set:

1. if at least one candidate achieves coverage $\geq 0.90$, select the
   candidate with the lowest Interval Score;
2. ties are broken by lower mean interval width;
3. if no candidate reaches 0.90 coverage, select the candidate with the
   highest coverage, then lower Interval Score, then lower mean width.

The Outer Test set is not used for parameter selection.

In [14]:
# Chapter 5.1 — Frozen GCCP search grid

bandwidth_grid = np.linspace(
    0.1,
    5.0,
    20
)

lambda_grid = np.linspace(
    0.0,
    1.0,
    20
)

print("Bandwidth grid:")
print(np.round(bandwidth_grid, 6))

print("\nLambda grid:")
print(np.round(lambda_grid, 6))

print(
    "\nTotal GCCP candidates:",
    len(bandwidth_grid) * len(lambda_grid)
)

assert len(bandwidth_grid) == 20
assert len(lambda_grid) == 20

Bandwidth grid:
[0.1      0.357895 0.615789 0.873684 1.131579 1.389474 1.647368 1.905263
 2.163158 2.421053 2.678947 2.936842 3.194737 3.452632 3.710526 3.968421
 4.226316 4.484211 4.742105 5.      ]

Lambda grid:
[0.       0.052632 0.105263 0.157895 0.210526 0.263158 0.315789 0.368421
 0.421053 0.473684 0.526316 0.578947 0.631579 0.684211 0.736842 0.789474
 0.842105 0.894737 0.947368 1.      ]

Total GCCP candidates: 400


In [15]:
# Chapter 5.2 — Inner GCCP bandwidth × lambda grid search

import time

# ---------------------------------------------------------
# Pre-sort calibration scores once.
# Weighted quantile always uses this same score ordering.
# ---------------------------------------------------------

score_sort_idx = np.argsort(
    calibration_scores_inner
)

scores_sorted = calibration_scores_inner[
    score_sort_idx
]

y_val_array = y_inner_val.to_numpy(
    dtype=float
)

pred_val_array = np.asarray(
    pred_inner_val,
    dtype=float
)

gccp_grid_results = []

t_start = time.time()

candidate_count = 0
total_candidates = (
    len(bandwidth_grid)
    * len(lambda_grid)
)

for bandwidth in bandwidth_grid:

    for lambda_weight in lambda_grid:

        candidate_count += 1

        # ---------------------------------------------
        # Joint geographic + causal distance
        # ---------------------------------------------

        d_joint = np.sqrt(
            lambda_weight * d_geo_inner**2
            +
            (1.0 - lambda_weight)
            * d_causal_inner**2
        )

        weights = np.exp(
            -0.5
            * (d_joint / bandwidth)**2
        )

        # ---------------------------------------------
        # Author-style weighted quantile
        # Vectorized over all validation observations
        # ---------------------------------------------

        weights_sorted = weights[
            :,
            score_sort_idx
        ]

        cumulative_weights = np.cumsum(
            weights_sorted,
            axis=1
        )

        total_weights = cumulative_weights[
            :,
            -1
        ]

        valid_rows = (
            np.isfinite(total_weights)
            &
            (total_weights > 0)
        )

        q = np.full(
            len(y_val_array),
            np.nan,
            dtype=float
        )

        if valid_rows.any():

            normalized_cumsum = (
                cumulative_weights[valid_rows]
                /
                total_weights[valid_rows, None]
            )

            reached = (
                normalized_cumsum
                >= (1.0 - ALPHA)
            )

            q_indices = np.argmax(
                reached,
                axis=1
            )

            q[valid_rows] = scores_sorted[
                q_indices
            ]

        # ---------------------------------------------
        # Prediction intervals
        # ---------------------------------------------

        lower = pred_val_array - q
        upper = pred_val_array + q

        finite_interval = (
            np.isfinite(lower)
            &
            np.isfinite(upper)
        )

        if finite_interval.all():

            coverage = np.mean(
                (y_val_array >= lower)
                &
                (y_val_array <= upper)
            )

            mean_width = np.mean(
                upper - lower
            )

            mean_score = np.mean(
                interval_score(
                    y_val_array,
                    lower,
                    upper,
                    alpha=ALPHA
                )
            )

        else:

            coverage = np.nan
            mean_width = np.nan
            mean_score = np.nan

        gccp_grid_results.append({
            "bandwidth": float(bandwidth),
            "lambda": float(lambda_weight),
            "coverage": float(coverage),
            "mean_width": float(mean_width),
            "interval_score": float(mean_score),
            "valid_intervals": int(
                finite_interval.sum()
            )
        })

    print(
        f"Finished bandwidth "
        f"{bandwidth:.4f} "
        f"({candidate_count}/{total_candidates})"
    )

elapsed = time.time() - t_start

gccp_grid_df = pd.DataFrame(
    gccp_grid_results
)

print(
    "\nGrid search elapsed:",
    f"{elapsed:.2f} sec"
)

print(
    "Candidates evaluated:",
    len(gccp_grid_df)
)

print(
    "Fully valid candidates:",
    (
        gccp_grid_df["valid_intervals"]
        == len(y_val_array)
    ).sum()
)

display(
    gccp_grid_df.head(10).round(6)
)

Finished bandwidth 0.1000 (20/400)
Finished bandwidth 0.3579 (40/400)
Finished bandwidth 0.6158 (60/400)
Finished bandwidth 0.8737 (80/400)
Finished bandwidth 1.1316 (100/400)
Finished bandwidth 1.3895 (120/400)
Finished bandwidth 1.6474 (140/400)
Finished bandwidth 1.9053 (160/400)
Finished bandwidth 2.1632 (180/400)
Finished bandwidth 2.4211 (200/400)
Finished bandwidth 2.6789 (220/400)
Finished bandwidth 2.9368 (240/400)
Finished bandwidth 3.1947 (260/400)
Finished bandwidth 3.4526 (280/400)
Finished bandwidth 3.7105 (300/400)
Finished bandwidth 3.9684 (320/400)
Finished bandwidth 4.2263 (340/400)
Finished bandwidth 4.4842 (360/400)
Finished bandwidth 4.7421 (380/400)
Finished bandwidth 5.0000 (400/400)

Grid search elapsed: 0.66 sec
Candidates evaluated: 400
Fully valid candidates: 390


,bandwidth,lambda,coverage,mean_width,interval_score,valid_intervals
0,0.1,0.000000,NaN,NaN,NaN,313
1,0.1,0.052632,NaN,NaN,NaN,314
2,0.1,0.105263,NaN,NaN,NaN,314
3,0.1,0.157895,NaN,NaN,NaN,314
4,0.1,0.210526,NaN,NaN,NaN,314
5,0.1,0.263158,NaN,NaN,NaN,314
6,0.1,0.315789,NaN,NaN,NaN,314
7,0.1,0.368421,NaN,NaN,NaN,314
8,0.1,0.421053,NaN,NaN,NaN,314
9,0.1,0.473684,NaN,NaN,NaN,314


In [16]:
# Chapter 5.3 — Select Inner GCCP parameters

TARGET_COVERAGE = 0.90

valid_grid = (
    gccp_grid_df
    .dropna(
        subset=[
            "coverage",
            "mean_width",
            "interval_score"
        ]
    )
    .copy()
)

coverage_ok = valid_grid[
    valid_grid["coverage"]
    >= TARGET_COVERAGE
].copy()

if len(coverage_ok) > 0:

    selection_mode = (
        "coverage >= 0.90; "
        "min Interval Score; "
        "tie -> min width"
    )

    ranked = coverage_ok.sort_values(
        by=[
            "interval_score",
            "mean_width",
            "bandwidth",
            "lambda"
        ],
        ascending=[
            True,
            True,
            True,
            True
        ]
    )

else:

    selection_mode = (
        "no candidate reached 0.90; "
        "max coverage; "
        "tie -> min Interval Score; "
        "tie -> min width"
    )

    ranked = valid_grid.sort_values(
        by=[
            "coverage",
            "interval_score",
            "mean_width",
            "bandwidth",
            "lambda"
        ],
        ascending=[
            False,
            True,
            True,
            True,
            True
        ]
    )

best_gccp_inner = ranked.iloc[0]

print("===== INNER GCCP SELECTION =====")

print("Selection mode:")
print(selection_mode)

print(
    "\nCandidates with coverage >= 0.90:",
    len(coverage_ok)
)

print(
    f"\nBest bandwidth      : "
    f"{best_gccp_inner['bandwidth']:.10f}"
)

print(
    f"Best lambda         : "
    f"{best_gccp_inner['lambda']:.10f}"
)

print(
    f"Validation coverage : "
    f"{best_gccp_inner['coverage']:.4f}"
)

print(
    f"Mean width          : "
    f"{best_gccp_inner['mean_width']:.4f}"
)

print(
    f"Interval Score      : "
    f"{best_gccp_inner['interval_score']:.4f}"
)

print("\nTop 10 candidates:")
display(
    ranked.head(10).round(6)
)

===== INNER GCCP SELECTION =====
Selection mode:
no candidate reached 0.90; max coverage; tie -> min Interval Score; tie -> min width

Candidates with coverage >= 0.90: 0

Best bandwidth      : 1.6473684211
Best lambda         : 0.3684210526
Validation coverage : 0.8571
Mean width          : 31.7009
Interval Score      : 63.7239

Top 10 candidates:


,bandwidth,lambda,coverage,mean_width,interval_score,valid_intervals
127,1.647368,0.368421,0.857143,31.700868,63.723914,315
143,1.905263,0.157895,0.857143,31.703077,63.726123,315
58,0.615789,0.947368,0.857143,30.970108,64.168736,315
149,1.905263,0.473684,0.857143,30.883773,64.179511,315
166,2.163158,0.315789,0.857143,30.856018,64.200826,315
77,0.873684,0.894737,0.857143,30.929246,64.244592,315
183,2.421053,0.157895,0.857143,30.866180,64.314948,315
114,1.389474,0.736842,0.857143,30.865028,64.487207,315
132,1.647368,0.631579,0.857143,30.870262,64.492441,315
19,0.100000,1.000000,0.857143,33.627096,80.647767,315


In [17]:
# Chapter 5.4 — Save Inner GCCP tuning results

GCCP_OUTPUT_DIR = (
    OUTPUT_DIR / "gccp"
)

GCCP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

grid_path = (
    GCCP_OUTPUT_DIR
    / "seattle_gccp_inner_grid_v1.csv"
)

best_path = (
    GCCP_OUTPUT_DIR
    / "seattle_gccp_inner_best_params_v1.csv"
)

gccp_grid_df.to_csv(
    grid_path,
    index=False
)

pd.DataFrame(
    [best_gccp_inner]
).to_csv(
    best_path,
    index=False
)

print("Saved:")
print(grid_path)
print(best_path)

print(
    "\n✓ Inner GCCP parameter tuning complete."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp\seattle_gccp_inner_grid_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp\seattle_gccp_inner_best_params_v1.csv

✓ Inner GCCP parameter tuning complete.


In [19]:
# Chapter 6.1 — Restore Final Outer experiment

from pathlib import Path
import numpy as np
import pandas as pd

FINAL_GCCM_WEIGHT_PATH = (
    GCCM_DIR
    / "seattle_gccm_causal_weights_outer_train_v1.csv"
)

assert FINAL_GCCM_WEIGHT_PATH.exists()

gccm_weights_final = pd.read_csv(
    FINAL_GCCM_WEIGHT_PATH
)

feature_order = ASpatial_FEATURES.copy()

gccm_weights_final = (
    gccm_weights_final
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

final_causal_weights = (
    gccm_weights_final["causal_weight"]
    .to_numpy(dtype=float)
)

print("===== FINAL GCCM CAUSAL WEIGHTS =====")

display(
    gccm_weights_final[
        [
            "feature",
            "causal_strength",
            "causal_weight"
        ]
    ].round(6)
)

print(
    "\nWeight sum:",
    final_causal_weights.sum()
)

assert np.isclose(
    final_causal_weights.sum(),
    1.0
)

# ---------------------------------------------------------
# Restore Outer data
# ---------------------------------------------------------

X_outer_train = X_model_all.iloc[
    outer_train_indices
].copy()

X_outer_cal = X_model_all.iloc[
    outer_calibration_indices
].copy()

X_outer_test = X_model_all.iloc[
    outer_test_indices
].copy()

y_outer_train = y_all.iloc[
    outer_train_indices
].copy()

y_outer_cal = y_all.iloc[
    outer_calibration_indices
].copy()

y_outer_test = y_all.iloc[
    outer_test_indices
].copy()

X_aspatial_outer_cal = X_aspatial_all.iloc[
    outer_calibration_indices
].copy()

X_aspatial_outer_test = X_aspatial_all.iloc[
    outer_test_indices
].copy()

loc_outer_cal = loc_all[
    outer_calibration_indices
]

loc_outer_test = loc_all[
    outer_test_indices
]

print("\nOuter shapes:")
print("Train       :", X_outer_train.shape)
print("Calibration :", X_outer_cal.shape)
print("Test        :", X_outer_test.shape)

print("\n✓ Final Outer experiment restored.")

===== FINAL GCCM CAUSAL WEIGHTS =====


,feature,causal_strength,causal_weight
0,bathrooms,0.143220,0.101417
1,sqft_living,0.270903,0.191833
2,sqft_lot,0.171303,0.121304
3,grade,0.310584,0.219932
4,condition,0.000000,0.000000
5,waterfront,0.164455,0.116455
6,view,0.296150,0.209711
7,age,0.055566,0.039348



Weight sum: 0.9999999999999998

Outer shapes:
Train       : (2100, 10)
Calibration : (450, 10)
Test        : (450, 10)

✓ Final Outer experiment restored.


In [20]:
# Chapter 6.2 — Final XGBoost predictor

from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

model_final = XGBRegressor(
    **XGBOOST_PARAMS
)

model_final.fit(
    X_outer_train,
    y_outer_train
)

pred_outer_cal = model_final.predict(
    X_outer_cal
)

pred_outer_test = model_final.predict(
    X_outer_test
)

final_mae = mean_absolute_error(
    y_outer_test,
    pred_outer_test
)

final_rmse = root_mean_squared_error(
    y_outer_test,
    pred_outer_test
)

final_r2 = r2_score(
    y_outer_test,
    pred_outer_test
)

print("===== FINAL PREDICTOR CHECK =====")

print(f"Test MAE  : {final_mae:.4f}")
print(f"Test RMSE : {final_rmse:.4f}")
print(f"Test R²   : {final_r2:.4f}")

print("\nFrozen baseline expected:")
print("MAE  ≈ 7.7265")
print("RMSE ≈ 12.3241")
print("R²   ≈ 0.8476")

===== FINAL PREDICTOR CHECK =====
Test MAE  : 7.7265
Test RMSE : 12.3241
Test R²   : 0.8476

Frozen baseline expected:
MAE  ≈ 7.7265
RMSE ≈ 12.3241
R²   ≈ 0.8476


In [21]:
# Chapter 6.3 — Final geographic and causal distances

from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

# Fit scaler on Outer Calibration only
similarity_scaler_final = StandardScaler()

Z_outer_cal = similarity_scaler_final.fit_transform(
    X_aspatial_outer_cal
)

Z_outer_test = similarity_scaler_final.transform(
    X_aspatial_outer_test
)

# GCCM-weighted causal distance
d_causal_final = causal_feature_distance(
    Z_outer_cal,
    Z_outer_test,
    final_causal_weights
)

# Geographic distance
d_geo_final = cdist(
    loc_outer_test,
    loc_outer_cal,
    metric="euclidean"
)

# Ordinary EUC distance, only for diagnostic comparison
d_euc_final = euclidean_feature_distance(
    Z_outer_cal,
    Z_outer_test
)

corr_final = np.corrcoef(
    d_euc_final.ravel(),
    d_causal_final.ravel()
)[0, 1]

print("===== FINAL DISTANCE MATRICES =====")

print(
    "Geographic:",
    d_geo_final.shape,
    "mean =", d_geo_final.mean()
)

print(
    "Causal:",
    d_causal_final.shape,
    "mean =", d_causal_final.mean()
)

print(
    "\nCorrelation EUC vs causal:",
    corr_final
)

assert d_geo_final.shape == (450, 450)
assert d_causal_final.shape == (450, 450)
assert np.isfinite(d_geo_final).all()
assert np.isfinite(d_causal_final).all()

print("\n✓ Final GCCP distances ready.")

===== FINAL DISTANCE MATRICES =====
Geographic: (450, 450) mean = 0.202637686621435
Causal: (450, 450) mean = 3.4409144455986476

Correlation EUC vs causal: 0.9642885416053905

✓ Final GCCP distances ready.


In [22]:
# Chapter 6.4 — FINAL OUTER TEST GCCP
# Hyperparameters were frozen using Inner Validation.

FINAL_BANDWIDTH = 1.6473684210526318
FINAL_LAMBDA = 0.3684210526315789

calibration_scores_final = np.abs(
    y_outer_cal.to_numpy(dtype=float)
    - pred_outer_cal
)

gccp_final = evaluate_gccp_inner(
    bandwidth=FINAL_BANDWIDTH,
    lambda_weight=FINAL_LAMBDA,
    d_geo=d_geo_final,
    d_causal=d_causal_final,
    calibration_scores=calibration_scores_final,
    validation_predictions=pred_outer_test,
    y_validation=y_outer_test,
    alpha=ALPHA
)

print("======================================")
print("       FINAL OUTER TEST — GCCP")
print("======================================")

print(f"Bandwidth       : {FINAL_BANDWIDTH:.6f}")
print(f"Lambda          : {FINAL_LAMBDA:.6f}")
print(f"Coverage        : {gccp_final['coverage']:.4f}")
print(f"Mean Width      : {gccp_final['mean_width']:.4f}")
print(f"Interval Score  : {gccp_final['interval_score']:.4f}")

print(
    "\nFinite q:",
    np.isfinite(gccp_final["q"]).sum(),
    "/",
    len(gccp_final["q"])
)

       FINAL OUTER TEST — GCCP
Bandwidth       : 1.647368
Lambda          : 0.368421
Coverage        : 0.8889
Mean Width      : 32.8870
Interval Score  : 49.0295

Finite q: 450 / 450


## Chapter 7 — Main experiment summary

The final Outer Test experiment compares GCCP against all frozen
baseline methods under the same predictor and data split.

GCCP produced narrower intervals and a lower Interval Score than GeoCP,
but its empirical coverage fell below the nominal 90% target.

Therefore, the Seattle main experiment provides evidence of an
efficiency–coverage trade-off rather than an unambiguous improvement
over GeoCP.

GCCP and GeoSIMCP-MND showed very similar performance, indicating that
the additional GCCM-derived causal weighting does not yet provide a
clear advantage over conventional feature-similarity localization on
this cross-sectional housing dataset.

No hyperparameters are modified after observing the Outer Test results.

In [23]:
# Chapter 7.1 — Freeze the Seattle main-experiment comparison

main_results = pd.DataFrame([
    {
        "method": "CP",
        "coverage": 0.9089,
        "mean_width": 35.2126,
        "interval_score": 57.7319
    },
    {
        "method": "GeoCP",
        "coverage": 0.9156,
        "mean_width": 35.7454,
        "interval_score": 55.9272
    },
    {
        "method": "GeoSIMCP-EUC",
        "coverage": 0.9156,
        "mean_width": 35.7454,
        "interval_score": 55.9272
    },
    {
        "method": "GeoSIMCP-MND",
        "coverage": 0.8933,
        "mean_width": 32.3386,
        "interval_score": 49.0971
    },
    {
        "method": "GCCP",
        "coverage": gccp_final["coverage"],
        "mean_width": gccp_final["mean_width"],
        "interval_score": gccp_final["interval_score"]
    }
])

display(main_results.round(4))

comparison_path = (
    GCCP_OUTPUT_DIR
    / "seattle_main_experiment_comparison_v1.csv"
)

main_results.to_csv(
    comparison_path,
    index=False
)

print("Saved:")
print(comparison_path)

print("\n✓ Seattle main experiment frozen.")

,method,coverage,mean_width,interval_score
0,CP,0.9089,35.2126,57.7319
1,GeoCP,0.9156,35.7454,55.9272
2,GeoSIMCP-EUC,0.9156,35.7454,55.9272
3,GeoSIMCP-MND,0.8933,32.3386,49.0971
4,GCCP,0.8889,32.8870,49.0295


Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp\seattle_main_experiment_comparison_v1.csv

✓ Seattle main experiment frozen.


In [24]:
# Chapter 7.2 — Relative performance of GCCP

gccp_row = main_results[
    main_results["method"] == "GCCP"
].iloc[0]

comparison_rows = []

for baseline in [
    "CP",
    "GeoCP",
    "GeoSIMCP-EUC",
    "GeoSIMCP-MND"
]:

    b = main_results[
        main_results["method"] == baseline
    ].iloc[0]

    comparison_rows.append({
        "baseline": baseline,

        "coverage_change_pp":
            100 * (
                gccp_row["coverage"]
                - b["coverage"]
            ),

        "width_change_pct":
            100 * (
                gccp_row["mean_width"]
                - b["mean_width"]
            ) / b["mean_width"],

        "interval_score_change_pct":
            100 * (
                gccp_row["interval_score"]
                - b["interval_score"]
            ) / b["interval_score"]
    })

gccp_relative = pd.DataFrame(
    comparison_rows
)

display(
    gccp_relative.round(3)
)

relative_path = (
    GCCP_OUTPUT_DIR
    / "seattle_gccp_relative_performance_v1.csv"
)

gccp_relative.to_csv(
    relative_path,
    index=False
)

,baseline,coverage_change_pp,width_change_pct,interval_score_change_pct
0,CP,-2.001,-6.604,-15.074
1,GeoCP,-2.671,-7.997,-12.333
2,GeoSIMCP-EUC,-2.671,-7.997,-12.333
3,GeoSIMCP-MND,-0.441,1.696,-0.138


## Chapter 8 — Causal-weight stability ablation

The main Outer-Test result is frozen and is not modified in this section.

The Inner-Fit and Final Outer-Train GCCM estimates showed only moderate
agreement in their normalized causal weights. In particular, some
features changed their retained/non-retained status between the two
causal-estimation samples.

This ablation evaluates whether GCCP performance is sensitive to the
estimated causal-weight vector.

All variants use exactly the same:

- Outer Calibration and Outer Test sets;
- final XGBoost predictor;
- calibration residuals;
- feature standardization;
- bandwidth;
- lambda;
- conformal quantile implementation.

Only the causal feature-weight vector is changed.

These analyses are diagnostic robustness checks and are not used to
re-select the main GCCP model.

In [25]:
# Chapter 8.1 — Construct causal-weight ablation variants

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Final Outer-Train weights
# ---------------------------------------------------------

w_final = final_causal_weights.copy()

# ---------------------------------------------------------
# 2. Inner-Fit weights
# ---------------------------------------------------------

INNER_WEIGHT_PATH = (
    GCCM_DIR
    / "seattle_gccm_causal_weights_inner_fit_v1.csv"
)

inner_weight_df = pd.read_csv(
    INNER_WEIGHT_PATH
)

inner_weight_df = (
    inner_weight_df
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

w_inner = inner_weight_df[
    "causal_weight"
].to_numpy(dtype=float)

# ---------------------------------------------------------
# 3. Stable-intersection weights
#
# Retain only features with non-zero GCCM strength
# in BOTH Inner Fit and Final Outer Train.
# Use Final Outer-Train strengths, then renormalize.
# ---------------------------------------------------------

inner_active = (
    inner_weight_df["causal_strength"]
    .to_numpy(dtype=float)
    > 0
)

final_active = (
    gccm_weights_final["causal_strength"]
    .to_numpy(dtype=float)
    > 0
)

stable_mask = (
    inner_active
    & final_active
)

stable_strength = (
    gccm_weights_final["causal_strength"]
    .to_numpy(dtype=float)
    * stable_mask
)

assert stable_strength.sum() > 0

w_stable = (
    stable_strength
    / stable_strength.sum()
)

# ---------------------------------------------------------
# 4. Equal weights
#
# Diagnostic reference only:
# removes GCCM feature weighting while preserving
# the SAME GCCP bandwidth and lambda.
# ---------------------------------------------------------

w_equal = (
    np.ones(len(feature_order))
    / len(feature_order)
)

# ---------------------------------------------------------
# Summary table
# ---------------------------------------------------------

weight_ablation_table = pd.DataFrame({
    "feature": feature_order,
    "Final_Outer": w_final,
    "Inner_Fit": w_inner,
    "Stable_Intersection": w_stable,
    "Equal": w_equal
})

display(
    weight_ablation_table.round(6)
)

print("\nStable features:")
print(
    weight_ablation_table.loc[
        stable_mask,
        "feature"
    ].tolist()
)

print("\nWeight sums:")
print(
    weight_ablation_table[
        [
            "Final_Outer",
            "Inner_Fit",
            "Stable_Intersection",
            "Equal"
        ]
    ].sum()
)

,feature,Final_Outer,Inner_Fit,Stable_Intersection,Equal
0,bathrooms,0.101417,0.158578,0.120135,0.125
1,sqft_living,0.191833,0.219766,0.227237,0.125
2,sqft_lot,0.121304,0.165200,0.143692,0.125
3,grade,0.219932,0.258749,0.260522,0.125
4,condition,0.000000,0.073367,0.000000,0.125
5,waterfront,0.116455,0.000000,0.000000,0.125
6,view,0.209711,0.124340,0.248415,0.125
7,age,0.039348,0.000000,0.000000,0.125



Stable features:
['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'view']

Weight sums:
Final_Outer            1.0
Inner_Fit              1.0
Stable_Intersection    1.0
Equal                  1.0
dtype: float64


In [26]:
# Chapter 8.2 — Evaluate causal-weight stability variants

weight_variants = {
    "Final_Outer_GCCM": w_final,
    "Inner_Fit_GCCM": w_inner,
    "Stable_Intersection": w_stable,
    "Equal_Weights": w_equal
}

weight_ablation_results = []

for name, weights_vector in weight_variants.items():

    # ---------------------------------------------
    # Causal feature distance
    # ---------------------------------------------

    d_variant = causal_feature_distance(
        Z_outer_cal,
        Z_outer_test,
        weights_vector
    )

    # Similarity to ordinary Euclidean distance
    corr_euc = np.corrcoef(
        d_euc_final.ravel(),
        d_variant.ravel()
    )[0, 1]

    # ---------------------------------------------
    # SAME frozen GCCP hyperparameters
    # ---------------------------------------------

    result = evaluate_gccp_inner(
        bandwidth=FINAL_BANDWIDTH,
        lambda_weight=FINAL_LAMBDA,
        d_geo=d_geo_final,
        d_causal=d_variant,
        calibration_scores=calibration_scores_final,
        validation_predictions=pred_outer_test,
        y_validation=y_outer_test,
        alpha=ALPHA
    )

    weight_ablation_results.append({
        "weight_variant": name,
        "coverage": result["coverage"],
        "mean_width": result["mean_width"],
        "interval_score": result["interval_score"],
        "corr_with_EUC_distance": corr_euc,
        "finite_q": np.isfinite(
            result["q"]
        ).sum()
    })

weight_ablation_results = pd.DataFrame(
    weight_ablation_results
)

print(
    "===== CAUSAL-WEIGHT STABILITY ABLATION ====="
)

display(
    weight_ablation_results.round(6)
)

===== CAUSAL-WEIGHT STABILITY ABLATION =====


,weight_variant,coverage,mean_width,interval_score,corr_with_EUC_distance,finite_q
0,Final_Outer_GCCM,0.888889,32.886995,49.029472,0.964289,450
1,Inner_Fit_GCCM,0.888889,32.959940,47.858980,0.725147,450
2,Stable_Intersection,0.888889,33.080763,48.106515,0.733438,450
3,Equal_Weights,0.891111,32.551891,50.072985,1.000000,450


In [27]:
# Chapter 8.3 — Difference relative to frozen Final GCCP

main_final = (
    weight_ablation_results[
        weight_ablation_results[
            "weight_variant"
        ] == "Final_Outer_GCCM"
    ]
    .iloc[0]
)

weight_ablation_change = (
    weight_ablation_results.copy()
)

weight_ablation_change[
    "coverage_change_pp_vs_final"
] = 100 * (
    weight_ablation_change["coverage"]
    - main_final["coverage"]
)

weight_ablation_change[
    "width_change_pct_vs_final"
] = 100 * (
    weight_ablation_change["mean_width"]
    - main_final["mean_width"]
) / main_final["mean_width"]

weight_ablation_change[
    "score_change_pct_vs_final"
] = 100 * (
    weight_ablation_change["interval_score"]
    - main_final["interval_score"]
) / main_final["interval_score"]

display(
    weight_ablation_change.round(4)
)

,weight_variant,coverage,mean_width,interval_score,corr_with_EUC_distance,finite_q,coverage_change_pp_vs_final,width_change_pct_vs_final,score_change_pct_vs_final
0,Final_Outer_GCCM,0.8889,32.8870,49.0295,0.9643,450,0.0000,0.0000,0.0000
1,Inner_Fit_GCCM,0.8889,32.9599,47.8590,0.7251,450,0.0000,0.2218,-2.3873
2,Stable_Intersection,0.8889,33.0808,48.1065,0.7334,450,0.0000,0.5892,-1.8825
3,Equal_Weights,0.8911,32.5519,50.0730,1.0000,450,0.2222,-1.0190,2.1283


In [28]:
# Chapter 8.4 — Save causal-weight stability ablation

weight_table_path = (
    GCCP_OUTPUT_DIR
    / "seattle_gccp_weight_vectors_ablation_v1.csv"
)

weight_results_path = (
    GCCP_OUTPUT_DIR
    / "seattle_gccp_weight_stability_ablation_v1.csv"
)

weight_ablation_table.to_csv(
    weight_table_path,
    index=False
)

weight_ablation_change.to_csv(
    weight_results_path,
    index=False
)

print("Saved:")
print(weight_table_path)
print(weight_results_path)

print(
    "\n✓ Causal-weight stability ablation complete."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp\seattle_gccp_weight_vectors_ablation_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp\seattle_gccp_weight_stability_ablation_v1.csv

✓ Causal-weight stability ablation complete.


In [29]:
# Chapter 9 — KNN8 adjacency sensitivity of final GCCP

KNN8_WEIGHT_PATH = (
    GCCM_DIR
    / "seattle_gccm_causal_weights_outer_train_knn8_v1.csv"
)

knn8_weight_df = pd.read_csv(
    KNN8_WEIGHT_PATH
)

knn8_weight_df = (
    knn8_weight_df
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

w_knn8 = (
    knn8_weight_df[
        "causal_weight"
    ]
    .to_numpy(dtype=float)
)

assert np.isclose(
    w_knn8.sum(),
    1.0
)

# Recompute only causal similarity
d_causal_knn8 = causal_feature_distance(
    Z_outer_cal,
    Z_outer_test,
    w_knn8
)

corr_euc_knn8 = np.corrcoef(
    d_euc_final.ravel(),
    d_causal_knn8.ravel()
)[0, 1]

gccp_knn8 = evaluate_gccp_inner(
    bandwidth=FINAL_BANDWIDTH,
    lambda_weight=FINAL_LAMBDA,
    d_geo=d_geo_final,
    d_causal=d_causal_knn8,
    calibration_scores=calibration_scores_final,
    validation_predictions=pred_outer_test,
    y_validation=y_outer_test,
    alpha=ALPHA
)

print(
    "===== DELAUNAY vs KNN8 GCCP SENSITIVITY ====="
)

graph_gccp_comparison = pd.DataFrame([
    {
        "graph": "Delaunay",
        "coverage":
            gccp_final["coverage"],
        "mean_width":
            gccp_final["mean_width"],
        "interval_score":
            gccp_final["interval_score"],
        "corr_EUC_causal":
            corr_final
    },
    {
        "graph": "Symmetric_KNN8",
        "coverage":
            gccp_knn8["coverage"],
        "mean_width":
            gccp_knn8["mean_width"],
        "interval_score":
            gccp_knn8["interval_score"],
        "corr_EUC_causal":
            corr_euc_knn8
    }
])

display(
    graph_gccp_comparison.round(6)
)

print(
    "\nFinite KNN8 q:",
    np.isfinite(
        gccp_knn8["q"]
    ).sum(),
    "/",
    len(gccp_knn8["q"])
)

===== DELAUNAY vs KNN8 GCCP SENSITIVITY =====


,graph,coverage,mean_width,interval_score,corr_EUC_causal
0,Delaunay,0.888889,32.886995,49.029472,0.964289
1,Symmetric_KNN8,0.886667,32.529287,48.538894,0.900384



Finite KNN8 q: 450 / 450


In [30]:
# Chapter 9.2 — Evaluate KNN8 adjacency sensitivity of final GCCP

# Only the causal-weight vector changes.
# Everything else remains frozen:
# predictor, calibration/test sets, bandwidth, lambda,
# geographic distance and conformal procedure.

d_causal_knn8 = causal_feature_distance(
    Z_outer_cal,
    Z_outer_test,
    w_knn8
)

# Diagnostic: how different is KNN8 causal distance from ordinary EUC?
corr_euc_knn8 = np.corrcoef(
    d_euc_final.ravel(),
    d_causal_knn8.ravel()
)[0, 1]

# Frozen GCCP parameters from Inner Validation
gccp_knn8 = evaluate_gccp_inner(
    bandwidth=FINAL_BANDWIDTH,
    lambda_weight=FINAL_LAMBDA,
    d_geo=d_geo_final,
    d_causal=d_causal_knn8,
    calibration_scores=calibration_scores_final,
    validation_predictions=pred_outer_test,
    y_validation=y_outer_test,
    alpha=ALPHA
)

graph_gccp_comparison = pd.DataFrame([
    {
        "graph": "Delaunay",
        "coverage": gccp_final["coverage"],
        "mean_width": gccp_final["mean_width"],
        "interval_score": gccp_final["interval_score"],
        "corr_EUC_causal": corr_final
    },
    {
        "graph": "Symmetric_KNN8",
        "coverage": gccp_knn8["coverage"],
        "mean_width": gccp_knn8["mean_width"],
        "interval_score": gccp_knn8["interval_score"],
        "corr_EUC_causal": corr_euc_knn8
    }
])

print(
    "===== DELAUNAY vs KNN8 GCCP SENSITIVITY ====="
)

display(
    graph_gccp_comparison.round(6)
)

print(
    "\nFinite KNN8 q:",
    np.isfinite(gccp_knn8["q"]).sum(),
    "/",
    len(gccp_knn8["q"])
)

===== DELAUNAY vs KNN8 GCCP SENSITIVITY =====


,graph,coverage,mean_width,interval_score,corr_EUC_causal
0,Delaunay,0.888889,32.886995,49.029472,0.964289
1,Symmetric_KNN8,0.886667,32.529287,48.538894,0.900384



Finite KNN8 q: 450 / 450


In [31]:
# Chapter 9.3 — Save adjacency sensitivity result

from pathlib import Path

GCCP_OUTPUT_DIR_FINAL = (
    Path(r"E:\GeoSIMCP_Reproduce\geoconformal-main")
    / "GCCP_Seattle"
    / "outputs"
    / "gccp"
)

GCCP_OUTPUT_DIR_FINAL.mkdir(
    parents=True,
    exist_ok=True
)

graph_gccp_path = (
    GCCP_OUTPUT_DIR_FINAL
    / "seattle_gccp_delaunay_vs_knn8_sensitivity_v1.csv"
)

graph_gccp_comparison.to_csv(
    graph_gccp_path,
    index=False
)

print("Saved:")
print(graph_gccp_path)

print(
    "\n✓ GCCP adjacency sensitivity complete."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccp\seattle_gccp_delaunay_vs_knn8_sensitivity_v1.csv

✓ GCCP adjacency sensitivity complete.
